In [22]:
import requests
import argparse
import time

SERVER_PATH = "http://api.italianlp.it"


def execute_term_extraction(doc_ids, configuration=None):
    #configuration = {
    #    'pos_start_term': ['c:S'],
    #    'pos_internal_term': ['c:E'],
    #    'pos_end_term': ['c:S','c:A'],
    #    'max_length_term': 4
    #}
    configuration = {   'pos_start_term':   ['c:S','c:A'],              # begin with a fine common name (diritto, legge) or an adjective (interdisciplinare, aggregato)
                        'pos_internal_term':['c:E','c:A','f:S','c:B'],  # continue with a preposition (del, da, su, verso) and other names
                        'pos_end_term':     ['c:S','c:A'],              # end with a fine common name 
                        'max_length_term':  3 
                    }
    #configuration = {'max_length_term': 3}
    if configuration is None:
        response = requests.post(SERVER_PATH + '/documents/term_extraction',
                                 # richiesta al server di fare la term extraction su una lista di id
                                 json={'doc_ids': doc_ids})
    else:
        response = requests.post(SERVER_PATH + '/documents/term_extraction',
                                 # richiesta al server di fare la term extraction su una lista di id
                                 json={'doc_ids': doc_ids,
                                       'configuration': configuration})
    term_extraction_id = response.json()['id']  # id dove controllare se l'operazione è conclusa
    while True:
        r = requests.get(SERVER_PATH + '/documents/term_extraction',
                         {'id': term_extraction_id})
        res = r.json()
        print('Waiting for results...')
        if res['status'] == "OK":  # se lo status è OK significa che l'estrazione terminologica è completata
            return res["terms"]  # altrimenti si fa uno sleep di un secondo prima di controllare nuovamente
        time.sleep(5)

def get_similarity(doc_ids):
    response = requests.get(SERVER_PATH + "/documents/similarity",
                             params= {"doc_id_1":doc_ids[0], 
                                    "doc_id_2":doc_ids[1]})
    print(response.json())

from pprint import pprint
import pandas as pd
terms = pd.DataFrame(execute_term_extraction([1758]))
#print(terms)

# Add a column to count the number of words in each term
terms['word_count'] = terms['term'].apply(lambda x: len(x.split()))

# Sort the DataFrame by the number of words in the "term" column
terms = terms.sort_values(by='word_count')

# Drop the helper column if not needed
terms = terms.drop(columns=['word_count'])

#terms = terms.to_csv("out.csv",index=False,sep='\t')
# List of stopwords
italian_stopwords = {
    "cosa", "uomo", "donna", "bambino", "gente", "tempo", "anno", "giorno", "notte", "ora",
    "momento", "mondo", "parte", "vita", "casa", "città", "paese", "strada",
    "voce", "nome", "lavoro", "punto", "modo", "parola", "caso", "guerra", "storia", "forza"
}
# Filter the DataFrame to exclude terms that are in the stopwords list
#terms = terms[~terms['term'].isin(italian_stopwords)]


print(terms)
terms.to_csv("terms.csv", sep="\t")
#get_similarity([1424, 1425])


Waiting for results...
Waiting for results...
                                       term  domain_relevance  frequency
0                                   diritto        100.000000         10
1                         Interdisciplinare         54.545455          2
5                                   cultura         36.363636          4
4                                     corso         36.363636          4
13                                 avvocati         18.181818          2
..                                      ...               ...        ...
3             Interdisciplinare nei docenti         36.363636          2
35                diritto anche processuale         18.181818          1
36                  ministero della cultura         18.181818          1
11  interdisciplinare anche nei destinatari         21.108437          1
10      progetto didattico del dipartimento         21.108437          1

[64 rows x 3 columns]


In [12]:
def wait_for_pos_tagging(doc_id):
    page = 1
    # inizializzazione dummy della risposta del server per poter scrivere la condizione del while
    api_res = {'postagging_executed': False, 'sentences': {'next': False, 'data': []}}
    while not api_res['postagging_executed'] or api_res['sentences']['next']:
        r = requests.get(SERVER_PATH + '/documents/details/%s?page=%s' % (doc_id, page))
        api_res = r.json()

        if not api_res['postagging_executed']:
            print('Waiting for pos tagging...')
            time.sleep(1)
            continue
        else:
            from pprint import pprint
            pprint(api_res)
            import json
            with open("res.json","w") as f:
                json.dump(api_res,f,indent=4)
            

        if api_res['sentences']['next']:
            page += 1

wait_for_pos_tagging(1729)

{'created_at': '2024-07-04T10:29:44.855620Z',
 'doc_time': '2024-07-04T10:29:44.855666Z',
 'hate_executed': False,
 'hate_no_probability': None,
 'hate_value': None,
 'hate_yes_probability': None,
 'irony_executed': False,
 'irony_no_probability': None,
 'irony_value': None,
 'irony_yes_probability': None,
 'language': 'IT',
 'named_entity_executed': False,
 'parsing_executed': False,
 'postagging_executed': True,
 'readability_executed': False,
 'readability_score_all': None,
 'readability_score_base': None,
 'readability_score_lexical': None,
 'readability_score_syntax': None,
 'sarcasm_executed': False,
 'sarcasm_no_probability': None,
 'sarcasm_value': None,
 'sarcasm_yes_probability': None,
 'sentences': {'count': 12,
               'data': [{'raw_text': 'Diritto alla bellezza è un corso online '
                                     'che nasce da un progetto didattico del '
                                     'dipartimento di diritto privato e '
                                  

In [7]:
terms.to_csv("out.csv",index=False,sep='\t')

stopwords = ["cosa"]

import networkx as nx
import matplotlib.pyplot as plt
%matplotlib


plt.close("all")
plt.figure(figsize=(30,20))
# Initialize a graph
G = nx.DiGraph()

# Add nodes
for index, row in terms.iterrows():
    G.add_node(index, term=row['term'], domain_relevance=row['domain_relevance'], frequency=row['frequency'])

# Add edges based on term composition
for i, row_i in terms.iterrows():
    for j, row_j in terms.iterrows():
        if i != j and row_i['term'] in row_j['term']:
            G.add_edge(i, j)

# Create a layout for our nodes
pos = nx.spring_layout(G, iterations=50,k=0.6)

# Draw the nodes
nx.draw_networkx_nodes(G, pos, node_color='#66ffff', node_size=600)

# Draw the edges
nx.draw_networkx_edges(G, pos)

# Draw the labels
labels = {index: "\n".join(row['term'].split(" ")) for index, row in terms.iterrows()}
nx.draw_networkx_labels(G, pos, labels, font_size=8)
plt.show()


Using matplotlib backend: TkAgg
